# Classificação supervisionada no GEE com polígonos separados

Leia antes [Classificação supervisionada e avaliação no GEE](../aulas/06_classificacao_e_validacao_no_gee.md).

Este notebook demonstra o fluxo completo de classificação pixel a pixel com `ee.Classifier.smileRandomForest` em Python:

1. Construir uma composta Sentinel-2 limpa com bandas ópticas, NDVI e NDWI (McFeeters) na região do Lago do Manso (MT).
2. Definir vários polígonos separados de treino e de teste por classe, sem sobreposição espacial.
3. Conferir a mediana de NDVI e NDWI de cada polígono antes do treino.
4. Extrair atributos com `sampleRegions` fixando projeção e escala (`EPSG:32721`, 20 m), treinar Random Forest e avaliar com matriz de confusão.

Aviso importante: os polígonos abaixo são amostras didáticas conferidas sobre a cena pública Sentinel-2 L2A `S2B_21LXD_20240902_0_L2A` (02/09/2024; Contains modified Copernicus Sentinel data 2024), selecionadas com raio de 120 m e pureza de classe SCL igual ou superior a 98%. Separar polígonos dentro da mesma cena reduz a autocorrelação imediata entre pixels vizinhos, mas pixels de poucos polígonos não formam uma amostra probabilística independente e a matriz resultante não estima a acurácia real do mapa inteiro.


In [ ]:
import os
import ee
import geemap

GEE_PROJECT_ID = os.environ.get("GEE_PROJECT_ID", "SEU_PROJECT_ID_AQUI")

# ee.Authenticate()
ee.Initialize(project=GEE_PROJECT_ID)


## 1. Composta Sentinel-2 na região do Lago do Manso (MT)

Filtramos o tile `21LXD` durante a estação seca de 2024 (`2024-07-01` a `2024-10-01`) com máscara de qualidade `SCL` e fator de escala `0.0001`.


In [ ]:
centro_manso = ee.Geometry.Point([-55.76, -14.92])
area_classificacao = centro_manso.buffer(18000).bounds()

CRS_NATIVO = "EPSG:32721"
ESCALA_MODELO = 20
PROJECAO_MODELO = ee.Projection(CRS_NATIVO).atScale(ESCALA_MODELO)

BANDAS_OPTICAS = ["B2", "B3", "B4", "B8", "B11", "B12"]
BANDAS_MODELO = ["B2", "B3", "B4", "B8", "B11", "B12", "NDVI", "NDWI"]


def preparar_s2(imagem):
    scl = imagem.select("SCL")
    mascara = (
        scl.neq(1)
        .And(scl.neq(3))
        .And(scl.neq(8))
        .And(scl.neq(9))
        .And(scl.neq(10))
        .And(scl.neq(11))
    )
    opticas = imagem.select(BANDAS_OPTICAS).multiply(0.0001).updateMask(mascara)
    return opticas.copyProperties(imagem, ["system:time_start"])


composta = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(area_classificacao)
    .filter(ee.Filter.eq("MGRS_TILE", "21LXD"))
    .filterDate("2024-07-01", "2024-10-01")
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 25))
    .map(preparar_s2)
    .median()
)

ndvi = composta.normalizedDifference(["B8", "B4"]).rename("NDVI")
ndwi = composta.normalizedDifference(["B3", "B8"]).rename("NDWI")
imagem_preditores = composta.addBands([ndvi, ndwi]).select(BANDAS_MODELO)


## 2. Polígonos separados de treino e teste

Legenda didática usada aqui (4 polígonos de treino e 3 de teste por classe, raio 120 m, espaçados por mais de 2 km, conferidos na cena `S2B_21LXD_20240902_0_L2A`):

- `0`: água (reservatório; SCL = 6, NDWI > +0,55, NDVI < -0,30)
- `1`: vegetação densa (SCL = 4, NDVI > +0,63, NDWI < -0,59)
- `2`: solo exposto / pastagem seca (SCL = 5, NDVI entre +0,11 e +0,16)

Inspecione os polígonos no mapa e confira a tabela espectral na célula seguinte antes de rodar o treinamento.


In [ ]:
RAIO_M = 120

poligonos_treino = ee.FeatureCollection([
    # Classe 0: água
    ee.Feature(ee.Geometry.Point([-55.7022, -14.9439]).buffer(RAIO_M), {"id": "tr_agua_1", "classe": 0, "conjunto": "treino"}),
    ee.Feature(ee.Geometry.Point([-55.6566, -14.9581]).buffer(RAIO_M), {"id": "tr_agua_2", "classe": 0, "conjunto": "treino"}),
    ee.Feature(ee.Geometry.Point([-55.7475, -14.9885]).buffer(RAIO_M), {"id": "tr_agua_3", "classe": 0, "conjunto": "treino"}),
    ee.Feature(ee.Geometry.Point([-55.8062, -14.9716]).buffer(RAIO_M), {"id": "tr_agua_4", "classe": 0, "conjunto": "treino"}),
    # Classe 1: vegetação densa
    ee.Feature(ee.Geometry.Point([-55.8759, -14.9710]).buffer(RAIO_M), {"id": "tr_veg_1", "classe": 1, "conjunto": "treino"}),
    ee.Feature(ee.Geometry.Point([-55.7858, -14.9552]).buffer(RAIO_M), {"id": "tr_veg_2", "classe": 1, "conjunto": "treino"}),
    ee.Feature(ee.Geometry.Point([-55.8631, -14.9357]).buffer(RAIO_M), {"id": "tr_veg_3", "classe": 1, "conjunto": "treino"}),
    ee.Feature(ee.Geometry.Point([-55.7865, -14.8251]).buffer(RAIO_M), {"id": "tr_veg_4", "classe": 1, "conjunto": "treino"}),
    # Classe 2: solo exposto / pastagem seca
    ee.Feature(ee.Geometry.Point([-55.7089, -14.9087]).buffer(RAIO_M), {"id": "tr_solo_1", "classe": 2, "conjunto": "treino"}),
    ee.Feature(ee.Geometry.Point([-55.7317, -14.8329]).buffer(RAIO_M), {"id": "tr_solo_2", "classe": 2, "conjunto": "treino"}),
    ee.Feature(ee.Geometry.Point([-55.8270, -14.9030]).buffer(RAIO_M), {"id": "tr_solo_3", "classe": 2, "conjunto": "treino"}),
    ee.Feature(ee.Geometry.Point([-55.6764, -14.9085]).buffer(RAIO_M), {"id": "tr_solo_4", "classe": 2, "conjunto": "treino"}),
])

poligonos_teste = ee.FeatureCollection([
    # Classe 0: água
    ee.Feature(ee.Geometry.Point([-55.7320, -14.9296]).buffer(RAIO_M), {"id": "te_agua_1", "classe": 0, "conjunto": "teste"}),
    ee.Feature(ee.Geometry.Point([-55.7618, -14.9189]).buffer(RAIO_M), {"id": "te_agua_2", "classe": 0, "conjunto": "teste"}),
    ee.Feature(ee.Geometry.Point([-55.6508, -14.9825]).buffer(RAIO_M), {"id": "te_agua_3", "classe": 0, "conjunto": "teste"}),
    # Classe 1: vegetação densa
    ee.Feature(ee.Geometry.Point([-55.8416, -14.9618]).buffer(RAIO_M), {"id": "te_veg_1", "classe": 1, "conjunto": "teste"}),
    ee.Feature(ee.Geometry.Point([-55.8194, -14.9346]).buffer(RAIO_M), {"id": "te_veg_2", "classe": 1, "conjunto": "teste"}),
    ee.Feature(ee.Geometry.Point([-55.8766, -15.0144]).buffer(RAIO_M), {"id": "te_veg_3", "classe": 1, "conjunto": "teste"}),
    # Classe 2: solo exposto / pastagem seca
    ee.Feature(ee.Geometry.Point([-55.6488, -14.8577]).buffer(RAIO_M), {"id": "te_solo_1", "classe": 2, "conjunto": "teste"}),
    ee.Feature(ee.Geometry.Point([-55.7688, -14.8439]).buffer(RAIO_M), {"id": "te_solo_2", "classe": 2, "conjunto": "teste"}),
    ee.Feature(ee.Geometry.Point([-55.7948, -14.8423]).buffer(RAIO_M), {"id": "te_solo_3", "classe": 2, "conjunto": "teste"}),
])

# Inspecione no mapa antes de treinar:
mapa_amostras = geemap.Map()
mapa_amostras.centerObject(area_classificacao, 11)
vis_rgb = {"bands": ["B4", "B3", "B2"], "min": 0.02, "max": 0.30, "gamma": 1.2}
mapa_amostras.addLayer(composta.clip(area_classificacao), vis_rgb, "Sentinel-2 RGB")
mapa_amostras.addLayer(poligonos_treino, {"color": "yellow"}, "Polígonos de treino")
mapa_amostras.addLayer(poligonos_teste, {"color": "magenta"}, "Polígonos de teste")
mapa_amostras


## 3. Conferência espectral dos polígonos antes do treino

Antes de ajustar qualquer classificador, verifique se os rótulos fazem sentido físico: água deve ter `NDWI` positivo e `NDVI` negativo; vegetação densa deve ter `NDVI` alto e `NDWI` negativo; solo/pastagem seca deve ficar com `NDVI` baixo e `NDWI` negativo.


In [ ]:
todos_poligonos = poligonos_treino.merge(poligonos_teste)

resumo_poligonos = imagem_preditores.select(["NDVI", "NDWI"]).reduceRegions(
    collection=todos_poligonos,
    reducer=ee.Reducer.median(),
    crs=CRS_NATIVO,
    scale=ESCALA_MODELO,
)

for feicao in resumo_poligonos.getInfo()["features"]:
    p = feicao["properties"]
    print(
        f"{p['id']:10s} | classe={p['classe']} | {p['conjunto']:6s} | "
        f"NDVI_med={p['NDVI']:+.3f} | NDWI_med={p['NDWI']:+.3f}"
    )


## 4. Amostragem, treino e matriz de confusão

Extraímos os pixels de treino e de teste separadamente na projeção nativa UTM (`EPSG:32721`) a 20 m (`scale=20`, compatível com as bandas SWIR `B11` e `B12`). Lembre-se de que a matriz abaixo avalia apenas os polígonos de teste escolhidos e não estima a acurácia probabilística de todo o mapa.


In [ ]:
amostras_treino = imagem_preditores.sampleRegions(
    collection=poligonos_treino,
    properties=["classe"],
    projection=PROJECAO_MODELO,
    scale=ESCALA_MODELO,
)

amostras_teste = imagem_preditores.sampleRegions(
    collection=poligonos_teste,
    properties=["classe"],
    projection=PROJECAO_MODELO,
    scale=ESCALA_MODELO,
)

print("Pixels de treino:", amostras_treino.size().getInfo())
print("Pixels de teste:", amostras_teste.size().getInfo())

classificador = ee.Classifier.smileRandomForest(
    numberOfTrees=100,
    seed=42,
).train(
    features=amostras_treino,
    classProperty="classe",
    inputProperties=BANDAS_MODELO,
)

teste_predito = amostras_teste.classify(classificador)
matriz = teste_predito.errorMatrix("classe", "classification")

print("Matriz de confusão nos polígonos de teste (linhas = referência, colunas = predito):")
print(matriz.getInfo())
print("Acurácia global (polígonos de teste):", round(matriz.accuracy().getInfo(), 4))
print("Acurácia do produtor:", matriz.producersAccuracy().getInfo())
print("Acurácia do usuário:", matriz.consumersAccuracy().getInfo())


## 5. Visualizar o mapa classificado

Compare o mapa classificado com a composição RGB para localizar confusões visuais (por exemplo, sombras de relevo confundidas com água ou margens rasas do reservatório).

Nota de escala: o modelo foi treinado com pixels de 20 m, mas em zoom 11 o `geemap` pede ao GEE uma classificação na escala da pirâmide da tela (cerca de 75 m). Ao dar zoom ou ao exportar o mapa (`Export.image.toDrive` com `crs="EPSG:32721"` e `scale=20`), a classificação ocorre na resolução de 20 m.


In [ ]:
mapa_classificado = imagem_preditores.classify(classificador).clip(area_classificacao)

vis_classes = {
    "min": 0,
    "max": 2,
    "palette": ["#1f78b4", "#238b45", "#d9a05b"],
}

mapa_resultado = geemap.Map()
mapa_resultado.centerObject(area_classificacao, 11)
mapa_resultado.addLayer(composta.clip(area_classificacao), vis_rgb, "Sentinel-2 RGB")
mapa_resultado.addLayer(mapa_classificado, vis_classes, "Classificação Random Forest")
mapa_resultado


## 6. Experimentos sugeridos

1. Remova `NDVI` e `NDWI` de `BANDAS_MODELO` e veja se a matriz de confusão muda quando o modelo usa apenas as seis bandas originais.
2. Faça um split aleatório por pixel dentro dos mesmos polígonos de treino (`randomColumn()`) e compare a acurácia obtida com a acurácia nos polígonos espacialmente separados.
3. Use `classificador.explain().getInfo()` para inspecionar a importância relativa das bandas no Random Forest.
